# Phase 5-6: Advanced Machine Learning

## 🎯 Learning Objectives

By the end of this notebook, you will:

- ✅ Master cross-validation techniques
- ✅ Tune hyperparameters effectively
- ✅ Understand and apply ensemble methods
- ✅ Use XGBoost and LightGBM
- ✅ Handle imbalanced datasets
- ✅ Build complete ML pipelines

**Time Required:** 2-3 weeks  
**Difficulty:** Advanced  
**Prerequisites:** Phases 0-4 completed

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, BaggingClassifier, VotingClassifier)

# Cross-validation
from sklearn.model_selection import (cross_val_score, KFold, StratifiedKFold,
                                     LeaveOneOut, GridSearchCV, RandomizedSearchCV,
                                     train_test_split, learning_curve)

# Preprocessing and pipelines
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Metrics
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, roc_auc_score, f1_score)

# Datasets
from sklearn.datasets import load_breast_cancer, make_classification

# Imbalanced learning
# Note: Run 'pip install imbalanced-learn' if not installed
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False
    print("⚠️ imbalanced-learn not installed. Run: pip install imbalanced-learn")

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("✅ All libraries imported successfully!")

---

# Part A: Cross-Validation

## 5.1 Why Cross-Validation?

**Problem with simple train-test split:**
- Results depend on which data goes to train vs test
- May not generalize well

**Solution:** Use multiple train-test splits and average the results!

In [ ]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset: {data.feature_names[:5]}...")
print(f"Shape: {X.shape}")
print(f"Classes: {np.unique(y)} (0=malignant, 1=benign)")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# K-Fold Cross-Validation

model = LogisticRegression(max_iter=1000)

# 5-Fold CV
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("=== 5-Fold Cross-Validation ===")
print(f"Fold scores: {scores.round(4)}")
print(f"Mean accuracy: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

# Visualize
plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), scores, color='steelblue', edgecolor='black')
plt.axhline(y=scores.mean(), color='red', linestyle='--', label=f'Mean: {scores.mean():.4f}')
plt.xlabel('Fold', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('5-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
plt.legend()
plt.ylim(0.9, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Stratified K-Fold (maintains class distribution in each fold)

from sklearn.model_selection import StratifiedKFold

# Compare regular KFold vs StratifiedKFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== KFold vs StratifiedKFold ===")
print("\nRegular KFold:")
for i, (train_idx, test_idx) in enumerate(kfold.split(X)):
    test_class_dist = np.bincount(y[test_idx])
    print(f"  Fold {i+1}: Class 0={test_class_dist[0]}, Class 1={test_class_dist[1]}")

print("\nStratified KFold:")
for i, (train_idx, test_idx) in enumerate(stratified_kfold.split(X, y)):
    test_class_dist = np.bincount(y[test_idx])
    print(f"  Fold {i+1}: Class 0={test_class_dist[0]}, Class 1={test_class_dist[1]}")

print("\n✅ StratifiedKFold maintains consistent class ratios!")

In [ ]:
# Compare multiple models with CV

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    results[name] = {'mean': scores.mean(), 'std': scores.std(), 'scores': scores}

# Display results
print("=== Model Comparison with 5-Fold CV ===")
for name, result in results.items():
    print(f"{name}: {result['mean']:.4f} (+/- {result['std']*2:.4f})")

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
names = list(results.keys())
means = [r['mean'] for r in results.values()]
stds = [r['std'] for r in results.values()]

bars = ax.bar(names, means, yerr=stds, capsize=5, color='steelblue', edgecolor='black')
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Comparison using 5-Fold CV', fontsize=14, fontweight='bold')
ax.set_ylim(0.85, 1.0)
ax.grid(True, alpha=0.3)

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{mean:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---

# Part B: Hyperparameter Tuning

## 5.2 GridSearchCV

In [ ]:
# GridSearchCV: Exhaustive search over parameter grid

from sklearn.model_selection import GridSearchCV

# Define parameter grid for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Calculate total combinations
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)

print(f"Parameter grid has {total_combinations} combinations")
print(f"With 5-fold CV: {total_combinations * 5} model fits")

In [ ]:
# Run GridSearchCV (using smaller grid for speed)

param_grid_small = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_small,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

grid_search.fit(X, y)

print("\n=== GridSearchCV Results ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

In [ ]:
# Analyze GridSearch results

results_df = pd.DataFrame(grid_search.cv_results_)

# Show top 10 configurations
top_results = results_df.nsmallest(10, 'rank_test_score')[[
    'params', 'mean_test_score', 'std_test_score', 'rank_test_score'
]]

print("Top 10 Parameter Configurations:")
print(top_results.to_string())

# Visualize effect of max_depth
pivot_depth = results_df.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_depth, annot=True, fmt='.4f', cmap='YlGn', cbar_kws={'label': 'Accuracy'})
plt.title('GridSearch Results: max_depth vs n_estimators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# RandomizedSearchCV: Random sampling from parameter distributions

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define parameter distributions
param_distributions = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=30,  # Number of random combinations to try
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X, y)

print("\n=== RandomizedSearchCV Results ===")
print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")

---

# Part C: Ensemble Methods

## 5.3 Bagging and Boosting

In [ ]:
# Bagging: Bootstrap Aggregating

# Create base model
base_tree = DecisionTreeClassifier(random_state=42)

# Create Bagging classifier
bagging = BaggingClassifier(
    estimator=base_tree,
    n_estimators=50,
    max_samples=0.8,  # Use 80% of samples for each tree
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# Compare single tree vs bagging
tree_scores = cross_val_score(base_tree, X, y, cv=5)
bagging_scores = cross_val_score(bagging, X, y, cv=5)

print("=== Bagging vs Single Decision Tree ===")
print(f"Single Tree: {tree_scores.mean():.4f} (+/- {tree_scores.std()*2:.4f})")
print(f"Bagging (50 trees): {bagging_scores.mean():.4f} (+/- {bagging_scores.std()*2:.4f})")
print(f"\n✅ Bagging reduces variance and improves stability!")

In [ ]:
# AdaBoost: Adaptive Boosting

adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Weak learner (stump)
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

ada_scores = cross_val_score(adaboost, X, y, cv=5)

print("=== AdaBoost ===")
print(f"Accuracy: {ada_scores.mean():.4f} (+/- {ada_scores.std()*2:.4f})")

# Compare learning rates
learning_rates = [0.01, 0.1, 0.5, 1.0, 2.0]
lr_results = []

for lr in learning_rates:
    ada = AdaBoostClassifier(n_estimators=50, learning_rate=lr, random_state=42)
    scores = cross_val_score(ada, X, y, cv=5)
    lr_results.append(scores.mean())

plt.figure(figsize=(10, 5))
plt.plot(learning_rates, lr_results, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('CV Accuracy', fontsize=12)
plt.title('AdaBoost: Effect of Learning Rate', fontsize=14, fontweight='bold')
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Gradient Boosting

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_scores = cross_val_score(gb, X, y, cv=5)

print("=== Gradient Boosting ===")
print(f"Accuracy: {gb_scores.mean():.4f} (+/- {gb_scores.std()*2:.4f})")

# Fit and show feature importance
gb.fit(X, y)

# Top 10 features
importance_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Importance': gb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='forestgreen')
plt.xlabel('Importance', fontsize=12)
plt.title('Gradient Boosting: Top 10 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Voting Classifier (Combine multiple models)

from sklearn.svm import SVC

# Define base models
models_for_voting = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
]

# Hard voting (majority vote)
hard_voting = VotingClassifier(estimators=models_for_voting, voting='hard')

# Soft voting (average probabilities) - usually better
soft_voting = VotingClassifier(estimators=models_for_voting, voting='soft')

# Compare all
print("=== Voting Classifier Comparison ===")
for name, model in models_for_voting + [('Hard Voting', hard_voting), ('Soft Voting', soft_voting)]:
    scores = cross_val_score(model, X, y, cv=5)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

---

# Part D: XGBoost and LightGBM

## 5.4 Modern Gradient Boosting Libraries

In [ ]:
# Install XGBoost and LightGBM if needed
# Run these in terminal if not installed:
# pip install xgboost lightgbm

try:
    import xgboost as xgb
    print(f"✅ XGBoost version: {xgb.__version__}")
    XGB_AVAILABLE = True
except ImportError:
    print("⚠️ XGBoost not installed. Run: pip install xgboost")
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    print(f"✅ LightGBM version: {lgb.__version__}")
    LGB_AVAILABLE = True
except ImportError:
    print("⚠️ LightGBM not installed. Run: pip install lightgbm")
    LGB_AVAILABLE = False

In [ ]:
# XGBoost Example

if XGB_AVAILABLE:
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        objective='binary:logistic',
        random_state=42,
        eval_metric='logloss'
    )
    
    xgb_scores = cross_val_score(xgb_model, X, y, cv=5)
    
    print("=== XGBoost ===")
    print(f"Accuracy: {xgb_scores.mean():.4f} (+/- {xgb_scores.std()*2:.4f})")
    
    # Fit and show feature importance
    xgb_model.fit(X, y)
    
    # Plot importance
    importance_df = pd.DataFrame({
        'Feature': data.feature_names,
        'Importance': xgb_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['Feature'], importance_df['Importance'], color='orange')
    plt.xlabel('Importance', fontsize=12)
    plt.title('XGBoost: Top 10 Feature Importances', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("XGBoost not available - skipping this cell")

In [ ]:
# LightGBM Example

if LGB_AVAILABLE:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    )
    
    lgb_scores = cross_val_score(lgb_model, X, y, cv=5)
    
    print("=== LightGBM ===")
    print(f"Accuracy: {lgb_scores.mean():.4f} (+/- {lgb_scores.std()*2:.4f})")
    
    # Fit and show feature importance
    lgb_model.fit(X, y)
    
    importance_df = pd.DataFrame({
        'Feature': data.feature_names,
        'Importance': lgb_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['Feature'], importance_df['Importance'], color='purple')
    plt.xlabel('Importance', fontsize=12)
    plt.title('LightGBM: Top 10 Feature Importances', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("LightGBM not available - skipping this cell")

In [ ]:
# Compare All Boosting Algorithms

all_models = {
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

if XGB_AVAILABLE:
    all_models['XGBoost'] = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
if LGB_AVAILABLE:
    all_models['LightGBM'] = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)

print("=== All Models Comparison ===")
comparison_results = {}

for name, model in all_models.items():
    scores = cross_val_score(model, X, y, cv=5)
    comparison_results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
names = list(comparison_results.keys())
means = [r['mean'] for r in comparison_results.values()]
stds = [r['std'] for r in comparison_results.values()]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor='black')
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Comparison of Boosting Algorithms', fontsize=14, fontweight='bold')
ax.set_ylim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='y')

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{mean:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---

# Part E: Handling Imbalanced Data

## 5.5 Techniques for Imbalanced Datasets

In [ ]:
# Create imbalanced dataset

X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    random_state=42
)

print("=== Imbalanced Dataset ===")
print(f"Class distribution: {np.bincount(y_imb)}")
print(f"Class 0: {np.sum(y_imb==0)/len(y_imb)*100:.1f}%")
print(f"Class 1: {np.sum(y_imb==1)/len(y_imb)*100:.1f}%")

# Visualize
plt.figure(figsize=(8, 5))
plt.bar(['Class 0', 'Class 1'], np.bincount(y_imb), color=['steelblue', 'orange'])
plt.ylabel('Count', fontsize=12)
plt.title('Imbalanced Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Problem with imbalanced data

X_train, X_test, y_train, y_test = train_test_split(X_imb, y_imb, test_size=0.2, random_state=42)

# Train standard model
rf_imb = RandomForestClassifier(n_estimators=100, random_state=42)
rf_imb.fit(X_train, y_train)
y_pred = rf_imb.predict(X_test)

print("=== Standard Model on Imbalanced Data ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\n⚠️ High accuracy but poor recall on minority class!")

In [ ]:
# Solution 1: Class Weights

rf_weighted = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',  # Automatically adjust weights
    random_state=42
)

rf_weighted.fit(X_train, y_train)
y_pred_weighted = rf_weighted.predict(X_test)

print("=== Class Weights (Balanced) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_weighted):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_weighted))

print("\n✅ Better recall on minority class!")

In [ ]:
# Solution 2: SMOTE (Synthetic Minority Over-sampling Technique)

if IMBLEARN_AVAILABLE:
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    
    print("=== SMOTE Resampling ===")
    print(f"Before: {np.bincount(y_train)}")
    print(f"After: {np.bincount(y_resampled)}")
    
    # Train on resampled data
    rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_smote.fit(X_resampled, y_resampled)
    y_pred_smote = rf_smote.predict(X_test)
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred_smote))
else:
    print("imbalanced-learn not installed - skipping SMOTE")

In [ ]:
# Compare all approaches

approaches = {
    'Standard': y_pred,
    'Class Weights': y_pred_weighted
}

if IMBLEARN_AVAILABLE:
    approaches['SMOTE'] = y_pred_smote

print("=== Comparison of Approaches ===")
print(f"{'Approach':<15} {'Accuracy':<10} {'F1 (minority)':<15} {'Recall (minority)':<15}")
print("-" * 55)

for name, preds in approaches.items():
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    recall = (confusion_matrix(y_test, preds)[1, 1] / 
              confusion_matrix(y_test, preds)[1].sum())
    print(f"{name:<15} {acc:.4f}     {f1:.4f}          {recall:.4f}")

---

# Part F: ML Pipelines

## 5.6 Building Complete Pipelines

In [ ]:
# Create a complete ML pipeline

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Create sample mixed data
np.random.seed(42)
n = 500

df_pipeline = pd.DataFrame({
    'Age': np.random.randint(18, 70, n),
    'Income': np.random.normal(50000, 20000, n),
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'City': np.random.choice(['NYC', 'LA', 'Chicago'], n),
    'Target': np.random.randint(0, 2, n)
})

# Add missing values
df_pipeline.loc[np.random.choice(n, 30), 'Age'] = np.nan
df_pipeline.loc[np.random.choice(n, 20), 'Income'] = np.nan

print("Sample Data:")
print(df_pipeline.head(10))
print(f"\nMissing values: {df_pipeline.isnull().sum().sum()}")

In [ ]:
# Define preprocessing for different column types

# Identify columns
numeric_features = ['Age', 'Income']
categorical_features = ['Education', 'City']

# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessor defined!")
print("\nPipeline structure:")
print("- Numeric: Impute (median) -> Scale (standard)")
print("- Categorical: Impute (mode) -> One-Hot Encode")

In [ ]:
# Complete pipeline with model

# Prepare data
X_pipe = df_pipeline.drop('Target', axis=1)
y_pipe = df_pipeline['Target']

# Create complete pipeline
complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Cross-validate the entire pipeline
cv_scores = cross_val_score(complete_pipeline, X_pipe, y_pipe, cv=5)

print("=== Complete Pipeline Cross-Validation ===")
print(f"CV Scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Fit the pipeline
complete_pipeline.fit(X_pipe, y_pipe)
print("\n✅ Pipeline fitted successfully!")

In [ ]:
# GridSearch with Pipeline

# Define parameter grid for the classifier within pipeline
param_grid_pipe = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_pipe = GridSearchCV(
    complete_pipeline,
    param_grid_pipe,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_pipe.fit(X_pipe, y_pipe)

print("=== GridSearch with Pipeline ===")
print(f"Best parameters: {grid_pipe.best_params_}")
print(f"Best CV score: {grid_pipe.best_score_:.4f}")

In [ ]:
# Make predictions with the pipeline

# New data (can have missing values!)
new_data = pd.DataFrame({
    'Age': [25, 45, None],
    'Income': [40000, None, 80000],
    'Education': ['Bachelor', 'PhD', 'Master'],
    'City': ['NYC', 'LA', 'Chicago']
})

print("New data for prediction:")
print(new_data)

# Predict using best pipeline
predictions = grid_pipe.predict(new_data)
probabilities = grid_pipe.predict_proba(new_data)

print("\nPredictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"  Sample {i+1}: Class {pred} (probability: {prob[pred]:.2f})")

---

## 📝 Practice Exercise

### Complete ML Pipeline Project

In [ ]:
# Exercise: Build a complete ML solution

# Load breast cancer dataset
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X_ex, y_ex = data.data, data.target

print(f"Dataset shape: {X_ex.shape}")
print(f"Target distribution: {np.bincount(y_ex)}")

# TODO:
# 1. Split data (80/20)
# 2. Create a pipeline with:
#    - StandardScaler
#    - XGBoost or RandomForest classifier
# 3. Use GridSearchCV to tune:
#    - n_estimators: [50, 100, 200]
#    - max_depth: [3, 5, 7]
# 4. Evaluate on test set
# 5. Show confusion matrix and classification report

print("\nComplete the exercise!")

---

## ✅ Phase Completion Checklist

- [ ] Apply K-Fold and Stratified K-Fold cross-validation
- [ ] Use GridSearchCV and RandomizedSearchCV
- [ ] Understand Bagging vs Boosting
- [ ] Apply Gradient Boosting, XGBoost, and LightGBM
- [ ] Handle imbalanced datasets with class weights and SMOTE
- [ ] Build complete ML pipelines

---

## 🎯 Key Takeaways

1. **Cross-Validation**: Always use CV for reliable model evaluation
2. **Hyperparameter Tuning**: Start with RandomSearch, then GridSearch
3. **Bagging**: Reduces variance (Random Forest)
4. **Boosting**: Reduces bias (XGBoost, LightGBM)
5. **Imbalanced Data**: Use class weights or SMOTE
6. **Pipelines**: Combine preprocessing and modeling for production

---

## 🎓 Congratulations!

You've completed the Machine Learning Learning Path!

**What's Next?**
- 🧠 Deep Learning with TensorFlow/PyTorch
- 📊 Natural Language Processing
- 🖼️ Computer Vision
- 🚀 MLOps and Model Deployment

**Keep practicing with real datasets!**
- Kaggle competitions
- UCI ML Repository
- Personal projects

**Happy Learning! 🚀**